# Lab 3: The Persistent Agent — Memory and Multi-turn State

**Series**: Agentic AI Science Playbook — Foundation Layer
**Goal**: Build an agent that remembers previous turns and maintains context.
**Time**: ~45 min.

## Memory Taxonomy

```
 Short-term  (in-context)  ── conversation history in the prompt
 Long-term   (external)    ── key facts stored in a dict or DB
 Episodic    (session log) ── structured record of this session's actions
```

## Learning Objectives

By the end of this lab, you will be able to:
- Implement three distinct memory layers: short-term, long-term, and episodic
- Build a multi-turn agent that maintains context across conversation turns
- Apply memory window compression to handle long conversations
- Understand why memory is essential for scientific research workflows

## Why This Matters for Scientists

Scientific research is inherently **multi-session**. A researcher doesn't start from scratch every day — they build on weeks of prior experiments, literature reviews, and hypotheses.

An agent without memory is like a research assistant with amnesia. It can answer one question, but it can't:
- Remember what you searched for yesterday
- Build on findings from the last conversation
- Track which hypotheses have been tested and which remain open

This lab gives your agents the ability to **accumulate knowledge over time** — the foundation for persistent scientific research assistants.

> **Real-world parallel**: Google's AI Co-Scientist uses episodic memory to track which hypotheses have been generated, tested, and validated across multi-day research campaigns. The three-layer memory architecture you'll build here is the same pattern.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Labs 0-2 |
| Concepts | Agent loop, tool execution |

**NVIDIA Connection**: Memory management becomes critical with **NVIDIA NIM** because Nemotron models have specific context window sizes. The memory compression technique you'll learn here ensures your agent stays within context limits while retaining the most important information — essential for long scientific research sessions.

### Install Dependencies
We install the OpenAI Python client, which provides a unified interface for both OpenAI and NVIDIA NIM endpoints.

In [ ]:
!pip install -q openai

### Connect to LLM
Same dual-provider setup for OpenAI or NVIDIA NIM. The code detects which API key is available and configures the client accordingly. This pattern is reused across all labs.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")

## Concept: Why Three Memory Layers?

Human researchers use different types of memory for different purposes:

| Memory Type | Human Analog | Agent Implementation | Persistence |
|-------------|-------------|---------------------|-------------|
| **Short-term** | Working memory (current conversation) | Message history list | Current session only |
| **Long-term** | Notebook of facts and preferences | Key-value dictionary | Across sessions |
| **Episodic** | Lab notebook (what I did and when) | Structured action log | Across sessions |

Each layer serves a different need:

```
Short-term: "What did the user just ask?"
Long-term:  "What domain does this user work in?"
Episodic:   "What tools did I use last time, and what did I find?"
```

> **Design principle**: Short-term memory gives coherence within a conversation. Long-term memory gives personalization. Episodic memory gives learning from experience.

### Build Short-term Memory
This class stores the conversation as a list of messages. It keeps only the last N turns to stay within the LLM's context window. Think of it as the agent's 'working memory' — what it can see right now.

## Short-term Memory — Conversation History

In [ ]:
class ConversationMemory:
    def __init__(self, system_prompt, max_turns=20):
        self.system_prompt = system_prompt
        self.max_turns = max_turns
        self.history = []
    def add_user(self, content): self.history.append({"role": "user", "content": content})
    def add_assistant(self, content): self.history.append({"role": "assistant", "content": content})
    def get_messages(self):
        recent = self.history[-(self.max_turns*2):]
        return [{"role": "system", "content": self.system_prompt}] + recent
    def __repr__(self): return f"ConversationMemory({len(self.history)} messages)"

### Test Short-term Memory
Create an instance with a scientific system prompt and verify it initializes correctly.

In [ ]:
SYSTEM = "You are a scientific research assistant. Build on previous findings in the conversation."
memory = ConversationMemory(SYSTEM)
print(memory)

## Concept: Long-term Memory as a Knowledge Store

Long-term memory stores **facts that persist across sessions**. For a scientific agent, this includes:

- **User domain**: "The user works on CRISPR gene editing in mammalian cells"
- **Preferences**: "The user prefers bullet-point summaries, max 5 items"
- **Domain knowledge**: "The target gene is BRCA1, the organism is Homo sapiens"
- **Research context**: "We're investigating off-target effects of Cas9"

In production, this would be a database (Redis, PostgreSQL, or a vector store). For this lab, we use a simple Python dictionary — but the pattern is identical.

### Build Long-term Memory
A simple key-value store for facts that persist across sessions. The agent can `remember()` facts and `recall()` them later. The `format_for_prompt()` method injects known facts into the system prompt.

## Long-term Memory — Key-Value Store

In [ ]:
class LongTermMemory:
    def __init__(self): self.store = {}
    def remember(self, key, value):
        self.store[key] = value
        print(f"  [LTM] {key!r} = {value[:60]!r}")
    def recall(self, key): return self.store.get(key)
    def format_for_prompt(self):
        if not self.store: return ""
        lines = ["[Long-term memory — known facts:]"]
        for k, v in self.store.items(): lines.append(f"  - {k}: {v}")
        return "\n".join(lines)
    def __repr__(self): return f"LongTermMemory({len(self.store)} facts)"

ltm = LongTermMemory()
ltm.remember("user_domain", "CRISPR gene editing in mammalian cells")
ltm.remember("preferred_style", "concise bullet points, max 5 items")
print(); print(ltm.format_for_prompt())

## Concept: Episodic Memory — The Agent's Lab Notebook

Episodic memory records **what the agent did, when, and what happened**. This is analogous to a scientist's lab notebook:

```
[2024-03-15 10:30] Searched PubMed for "CRISPR off-target mammalian"
  → Found 3 relevant papers
[2024-03-15 10:32] Summarized findings on off-target detection methods
  → Key finding: GUIDE-seq detects 95% of off-target sites
[2024-03-15 10:35] User asked about base editors
  → Provided comparison of CBE vs ABE off-target profiles
```

This enables:
- **Session recap**: "Here's what we covered so far"
- **Debugging**: "Why did the agent choose that tool?"
- **Learning**: "Last time this query failed, so try a different approach"

### Build Episodic Memory
Records what the agent did, when, and what happened — like a lab notebook. Each episode captures the user's question, which tool was used, and a preview of the result.

## Episodic Memory — Session Log

In [ ]:
from datetime import datetime

class EpisodicMemory:
    def __init__(self): self.episodes = []
    def log(self, user_message, tool, result):
        self.episodes.append({
            "ts": datetime.now().isoformat(timespec="seconds"),
            "user_message": user_message,
            "tool_used": tool,
            "result_preview": (result or "")[:80],
        })
    def format_for_prompt(self, n=3):
        if not self.episodes: return ""
        lines = ["[Session log — recent actions:]"]
        for ep in self.episodes[-n:]:
            lines.append(f"  [{ep['ts']}] asked: {ep['user_message'][:50]}...")
            lines.append(f"    tool={ep['tool_used']} | {ep['result_preview']}")
        return "\n".join(lines)
    def __repr__(self): return f"EpisodicMemory({len(self.episodes)} episodes)"

episodic = EpisodicMemory()

## Persistent Agent — All Three Layers

### Combine All Three Memory Layers
This function enriches the system prompt with long-term facts and episodic history, then runs the conversation through the LLM. All three memory layers work together to give the agent full context.

In [ ]:
def run_persistent_agent(user_message, memory, ltm, episodic):
    enriched = memory.system_prompt
    ltm_block = ltm.format_for_prompt()
    ep_block = episodic.format_for_prompt()
    if ltm_block: enriched += f"\n\n{ltm_block}"
    if ep_block: enriched += f"\n\n{ep_block}"
    messages = [{"role": "system", "content": enriched}] + memory.history
    r = client.chat.completions.create(
        model=MODEL, temperature=0.3, max_tokens=300, messages=messages)
    reply = (r.choices[0].message.content or "").strip()
    memory.add_user(user_message)
    memory.add_assistant(reply)
    episodic.log(user_message, tool="conversational", result=reply)
    return reply

### Run a Multi-turn Research Session
Let's test our persistent agent with a 4-turn conversation about CRISPR. Watch how the agent builds on previous turns — referencing earlier findings and maintaining coherent dialogue.

In [ ]:
memory = ConversationMemory(SYSTEM)
ltm = LongTermMemory()
episodic = EpisodicMemory()
ltm.remember("user_domain", "CRISPR gene editing in mammalian cells")
ltm.remember("preferred_style", "concise bullet points")

conversation = [
    "What are the main off-target effects of CRISPR-Cas9?",
    "Which of those is most studied in recent literature?",
    "How do base editors compare to Cas9 in off-target risk?",
    "Summarize what we have covered so far.",
]
print("=== Multi-turn Research Session ===\n")
for turn, msg in enumerate(conversation, 1):
    print(f"Turn {turn} — User: {msg}")
    reply = run_persistent_agent(msg, memory, ltm, episodic)
    print(f"Turn {turn} — Agent: {reply[:250]}\n")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Multi-turn Research Session ===

  [LTM] 'user_domain' = 'CRISPR gene editing in mammalian cells'
  [LTM] 'preferred_style' = 'concise bullet points'

Turn 1 — User: What are the main off-target effects of CRISPR-Cas9?
Turn 1 — Agent: The main off-target effects of CRISPR-Cas9 include:
1. Unintended double-strand breaks at genomic sites...
2. Chromosomal rearrangements...

Turn 2 — User: Which of those is most studied in recent literature?
Turn 2 — Agent: Unintended cleavage at off-target genomic sites is the most extensively studied...

Turn 3 — User: How do base editors compare to Cas9 in off-target risk?
Turn 3 — Agent: Base editors (CBEs and ABEs) generally show lower off-target DNA editing...

Turn 4 — User: Summarize what we have covered so far.
Turn 4 — Agent: In this session, we explored CRISPR-Cas9 off-target effects...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: Memory Compression — Handling Long Conversations

LLMs have a finite context window (e.g., 8K, 32K, or 128K tokens). In a long research session, conversation history can exceed this limit.

**Memory compression** solves this by summarizing older messages:

```
Turn 1-10: [detailed messages]  →  [3-bullet summary]
Turn 11-14: [kept as-is for recent context]
```

This is a tradeoff:
- **Compression loses detail** but retains key findings
- **Keeping everything** risks exceeding the context window
- **The sweet spot**: summarize older turns, keep recent turns verbatim

> **NVIDIA NIM tip**: Nemotron models support up to 32K context. For longer research sessions, aggressive compression (keeping only the last 4 turns + summary) keeps the agent responsive without losing critical findings.

### Memory Window Compression
When conversations get long, we summarize older messages to stay within the context limit. This cell compresses early messages into bullet points while keeping recent messages intact.

## Memory Window Compression

In [ ]:
def compress_memory(memory, keep_last=4):
    if len(memory.history) <= keep_last * 2: return memory
    to_compress = memory.history[:-(keep_last*2)]
    text = "\n".join(f"{m['role'].upper()}: {m['content'][:200]}" for m in to_compress)
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=200,
        messages=[{"role": "user", "content":
            f"Summarize this conversation in 3-5 bullet points focusing on scientific findings:\n\n{text}"}])
    summary = (r.choices[0].message.content or "").strip()
    new_mem = ConversationMemory(memory.system_prompt, memory.max_turns)
    new_mem.history = [{"role": "system", "content": f"[Summary]\n{summary}"}] + memory.history[-(keep_last*2):]
    return new_mem

print(f"Before: {len(memory.history)} messages")
compressed = compress_memory(memory, keep_last=2)
print(f"After : {len(compressed.history)} messages")

<details>
<summary>Expected output (click to expand)</summary>

```
Before: 8 messages
After : 5 messages
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **What information would be most valuable** in long-term memory for an agent in your scientific domain? Think about user preferences, domain facts, and research context.
2. **How would you handle conflicting memories?** For example, if long-term memory says "user prefers bullet points" but the user just asked for a paragraph summary?
3. **Episodic memory enables learning.** How could an agent use its action log to avoid repeating failed searches or to improve its tool selection over time?

## Summary

| Type | Implementation | Use |
|------|---------------|-----|
| Short-term | Conversation history list | Coherent dialogue |
| Long-term | Key-value dict | User preferences, domain facts |
| Episodic | Structured action log | Session recap, debugging |

**Next**: Lab 4 — LangGraph for multi-step orchestrated workflows.

---
*Agentic AI Science Playbook — Foundation Layer, Lab 3.*